# KNEEP_2D on Thermodynamically Consistent Lattice ABP (occupancy + angle)

This notebook trains the repository's direct 2D KNEEP implementation,
`models.NEEP_K_2DF.MultiScaleK_2DF`, on the MIPS state point used by
`data/lattice_abp_tc/mips_demo.ipynb`.  The physical parameters are copied
unchanged: $L=16$, $G=32$, $\sigma=0.5$, $\phi=0.30$, $v_0=50$,
$D_r=1.5$, $D_t=1$, and $dt=10^{-4}$.

The model is trained **without EP labels**.  The simulator separately records
the accepted-hop medium entropy production

$$
\Delta S_{\rm med}=\frac{\mathbf v\cdot\Delta\mathbf r-\mu\Delta V}{D_t}
$$

at each departure lattice site.  Those exact pathwise increments are used only
after model selection, for held-out total and local comparisons.

Important interpretation: the default observation is occupancy only.  It hides
particle orientation and all microscopic hops between two saved frames.  Data
processing bounds the **ensemble-mean observed path KL** by the microscopic
path KL; at steady state the mean system-entropy rate vanishes, allowing a
mean-rate comparison with microscopic medium EP.  A finite-capacity two-frame
KNEEP estimate, individual pairs, cumulative finite segments, and local pixels
do not automatically obey that bound.  They are shown here as diagnostics, not
as equal labels.  Set `OBSERVATION_MODE="occupancy_polarization"` to add
$\rho\cos\theta$ and $\rho\sin\theta$ channels.  In the angle notebook those
channels condition the learned force, while its final medium-EP contraction is
kept on $\Delta\rho$ only; `save_interval=100` still marginalizes the
intermediate path.

## 0. Environment

Run this notebook in the same CUDA-enabled environment as `mips_demo.ipynb`.
Install the packages in `data/lattice_abp_tc/requirements-l40s.txt`, install a
CUDA PyTorch wheel separately, and make sure `nvcc`, a host C++ compiler, and
Ninja are available.

Every accepted transition in this thermodynamically consistent dynamics must
have a finite log rate ratio.  The Torch, Numba, and fused CUDA samplers use
half-open inverse-CDF bins (`draw < cumulative`), so a zero-probability
direction cannot be selected even when the uniform draw is exactly zero.
Non-finite exact EP is therefore a fatal implementation or stale-cache error.
**Restart an already-running Jupyter kernel after updating the repository**,
because a previously loaded CUDA extension remains cached in that process.

In [ ]:
from pathlib import Path
import hashlib
import importlib.util
import json
import math
import os
import platform
import re
import shutil
import subprocess
import sys
import time
import warnings
from argparse import Namespace
from datetime import datetime


def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for path in [start, *start.parents]:
        if (path / "data" / "lattice_abp_tc" / "core.py").exists():
            return path
    raise RuntimeError(
        "Could not find CNEEP_v2. Start Jupyter inside the repository or set the working directory."
    )


ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Set extension build variables before importing torch.
bootstrap_nvcc = shutil.which("nvcc.exe" if os.name == "nt" else "nvcc")
if bootstrap_nvcc and "CUDA_HOME" not in os.environ:
    os.environ["CUDA_HOME"] = str(Path(bootstrap_nvcc).resolve().parents[1])
os.environ.setdefault("TORCH_CUDA_ARCH_LIST", "8.9")
os.environ.setdefault("MAX_JOBS", str(min(8, os.cpu_count() or 1)))

required = ("numpy", "matplotlib", "tqdm", "ninja", "setuptools", "torch")
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError(
        "Missing notebook packages: " + ", ".join(missing)
        + ". Install data/lattice_abp_tc/requirements-l40s.txt and CUDA PyTorch."
    )

print("repo:", ROOT)
print("python:", sys.executable)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import display
from torch.utils.cpp_extension import CUDA_HOME, is_ninja_available
from tqdm.auto import tqdm


if not torch.cuda.is_available():
    raise RuntimeError(
        f"CUDA is unavailable (torch={torch.__version__}, wheel CUDA={torch.version.cuda}). "
        "Select the CUDA-enabled Jupyter kernel."
    )

DEVICE = torch.device("cuda:0")
gpu = torch.cuda.get_device_properties(DEVICE)
capability = torch.cuda.get_device_capability(DEVICE)
if capability != (8, 9):
    warnings.warn(
        f"The reference target is an L40S (sm_89); found {gpu.name} "
        f"with sm_{capability[0]}{capability[1]}."
    )

nvcc_name = "nvcc.exe" if os.name == "nt" else "nvcc"
nvcc = None
if CUDA_HOME is not None:
    candidate = Path(CUDA_HOME) / "bin" / nvcc_name
    if candidate.is_file():
        nvcc = str(candidate)
if nvcc is None:
    nvcc = shutil.which(nvcc_name)
host_compiler = (
    shutil.which("cl.exe")
    if os.name == "nt"
    else (shutil.which("c++") or shutil.which("g++") or shutil.which("clang++"))
)

toolchain_errors = []
if torch.version.cuda is None:
    toolchain_errors.append("Installed PyTorch is CPU-only.")
if CUDA_HOME is None:
    toolchain_errors.append("CUDA_HOME was not detected by torch.utils.cpp_extension.")
if nvcc is None:
    toolchain_errors.append("nvcc was not found; install the CUDA Toolkit.")
if host_compiler is None:
    toolchain_errors.append("No compatible host C++ compiler was found on PATH.")
if not is_ninja_available():
    toolchain_errors.append("Ninja is unavailable.")
if toolchain_errors:
    raise RuntimeError("Fused CUDA prerequisites failed:\n- " + "\n- ".join(toolchain_errors))

from data.lattice_abp_tc._cuda_backend import (
    cuda_backend_buildable,
    load_cuda_backend,
)

if not cuda_backend_buildable():
    raise RuntimeError("The fused CUDA backend is not buildable in this kernel.")
cuda_extension = load_cuda_backend(verbose=True)

from data.lattice_abp_tc import ThermodynamicLatticeABP, ThermodynamicLatticeABPParams
from models.NEEP_K_2D import MultiScaleK_2D
from models.NEEP_K_2DF import MultiScaleK_2DF

print(f"GPU: {gpu.name} ({gpu.total_memory / 2**30:.1f} GiB)")
print(f"PyTorch: {torch.__version__}, wheel CUDA: {torch.version.cuda}")
print(f"CUDA extension: {cuda_extension.__name__}")
print(f"platform: {platform.platform()}")

## 1. MIPS state point and KNEEP configuration

In [ ]:
def n_from_packing_fraction(phi, box_size, sigma):
    return int(round(phi * 4.0 * box_size * box_size / (math.pi * sigma * sigma)))


# --------------------------- user switches ---------------------------
REUSE_TRAJECTORY = True
ANGLE_OBSERVATION = True
OBSERVATION_MODE = "occupancy_polarization" if ANGLE_OBSERVATION else "occupancy"

# ---------------- MIPS demo physical parameters: unchanged ----------------
L = 16.0
GRID_SIZE = 32
SIGMA = 0.5
PHI = 0.30
N = n_from_packing_fraction(PHI, L, SIGMA)

physics = dict(
    N=N,
    L=L,
    grid_size=GRID_SIZE,
    sigma=SIGMA,
    epsilon=1.0,
    mobility=1.0,
    v0=50.0,
    Dr=1.5,
    Dt=1.0,
    dt=1.0e-4,
    prefactor="cv",
    strict_probabilities=True,
    probability_tolerance=1.0e-6,
    shuffle_particles=True,
    seed=7,
    device=str(DEVICE),
    dtype="float32",
    backend="cuda_fused",
)

# Run all replicas together. Each replica supplies 1,000 saved intervals:
# 80 train + 20 validation + 1 held-out test trajectory.
B = 101
N_TRAIN_ENSEMBLES = 80
N_VALIDATION_ENSEMBLES = 20
N_TEST_ENSEMBLES = 1
burn_in = 100_000
n_steps = 100_000
save_interval = 100

if OBSERVATION_MODE not in {"occupancy", "occupancy_polarization"}:
    raise ValueError(f"Unknown OBSERVATION_MODE={OBSERVATION_MODE!r}")
if B != N_TRAIN_ENSEMBLES + N_VALIDATION_ENSEMBLES + N_TEST_ENSEMBLES:
    raise ValueError("B must equal train + validation + test ensemble counts.")
if N_TEST_ENSEMBLES != 1:
    raise ValueError("This notebook expects exactly one held-out test trajectory.")
if n_steps <= 0 or save_interval <= 0 or n_steps % save_interval != 0:
    raise ValueError("n_steps must be positive and divisible by save_interval.")
if n_steps // save_interval < 4:
    raise ValueError("Save at least four trajectory intervals.")

opt = Namespace()
opt.model_type = "MultiScaleK_2DF" if ANGLE_OBSERVATION else "MultiScaleK_2D"
opt.device = str(DEVICE)
opt.alpha = -0.5
opt.beta = 1.0
opt.positional = False
opt.seq_len = 2
opt.max_distance = 5
opt.include_k0 = True
opt.n_channel = 32
opt.n_hidden = 2
opt.n_components = 1 if OBSERVATION_MODE == "occupancy" else 3
opt.input_shape = (GRID_SIZE, GRID_SIZE)
if ANGLE_OBSERVATION:
    # Match the effective settings in Corr_LatticeABP_TC.ipynb.
    opt.k_kernel_geometry = "euclidean"
    # Polarization channels condition the learned force, but medium EP is
    # contracted only with the occupancy displacement, not with Δtheta-like
    # polarization changes.
    opt.ep_component_indices = (0,)
opt.n_iter = 5_000 if ANGLE_OBSERVATION else 2_000
opt.train_batch_size = 256
opt.eval_batch_size = 256
opt.lr = 1.0e-2 if ANGLE_OBSERVATION else 1.0e-4
opt.wd = 1.0e-5
opt.clip_norm = 1.0
opt.record_freq = 100
opt.seed = 3

torch.manual_seed(opt.seed)
np.random.seed(opt.seed)

dt_saved = save_interval * physics["dt"]
params = ThermodynamicLatticeABPParams(**physics)
sim = ThermodynamicLatticeABP(params)
assert sim.backend == "cuda_fused" and sim.device.type == "cuda"

RESULT_ROOT = ROOT / "results"
run_tag = datetime.now().strftime("%Y-%m-%d-%H%M%S")
run_label = "CorrLABP-TC-KNEEP2D-angle" if ANGLE_OBSERVATION else "CorrLABP-TC-KNEEP2D"
current_result_folder = RESULT_ROOT / f"{run_label}-{run_tag}"
current_result_folder.mkdir(parents=True, exist_ok=False)
best_checkpoint_path = current_result_folder / "best_model_parameter.pth.tar"
current_checkpoint_path = current_result_folder / "model_parameter.pth.tar"

SAMPLER_SOURCE_FILES = (
    "data/lattice_abp_tc/core.py",
    "data/lattice_abp_tc/_numba_backend.py",
    "data/lattice_abp_tc/_cuda_backend.py",
    "data/lattice_abp_tc/csrc/cuda_sweep.cpp",
    "data/lattice_abp_tc/csrc/cuda_sweep_kernel.cu",
)
sampler_source_sha256 = hashlib.sha256(
    b"".join((ROOT / path).read_bytes() for path in SAMPLER_SOURCE_FILES)
).hexdigest()
trajectory_config = {
    "cache_version": 2,  # Exact local-EP dataset schema/source version.
    "sampler_source_sha256": sampler_source_sha256,
    "physics": {k: v for k, v in physics.items() if k != "device"},
    "B": B,
    "burn_in": burn_in,
    "n_steps": n_steps,
    "save_interval": save_interval,
}
config_json = json.dumps(trajectory_config, sort_keys=True)
cache_id = hashlib.sha256(config_json.encode("utf-8")).hexdigest()[:16]
CACHE_DIR = ROOT / "output" / "lattice_abp_tc_kneep"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
trajectory_cache = CACHE_DIR / f"trajectory_{cache_id}.pt"

print(
    f"L={params.L:g}, G={params.grid_size}, dl={params.dl:g}, N={params.N}, "
    f"phi={params.phi:.6f}, Pe={params.Pe:.3f}, gamma2={params.gamma2:.1f}"
)
print(
    f"B={B}, burn_in={burn_in}, n_steps={n_steps}, "
    f"save_interval={save_interval}, dt_saved={dt_saved:g}"
)
print(f"observation={OBSERVATION_MODE}, KNEEP components={opt.n_components}")
print("trajectory cache:", trajectory_cache)
print("results:", current_result_folder)

## 2. Equilibrated trajectory with exact local medium EP

In [ ]:
# Recreate the simulator immediately before the production run so the cache
# provenance explicitly corresponds to physics["seed"] == 7.
sim = ThermodynamicLatticeABP(params)
trajectory_was_generated = False

tensor_keys = (
    "sites",
    "theta",
    "times",
    "occupancy",
    "accepted_hops",
    "exact_medium_ep",
    "exact_active_medium_ep",
    "exact_wca_medium_ep",
    "exact_medium_ep_rate",
    "exact_medium_ep_maps",
)

if REUSE_TRAJECTORY and trajectory_cache.exists():
    try:
        result = torch.load(trajectory_cache, map_location="cpu", weights_only=True)
    except TypeError:  # PyTorch < 2.0
        result = torch.load(trajectory_cache, map_location="cpu")
    missing_keys = sorted(set(tensor_keys) - set(result))
    if missing_keys:
        raise RuntimeError(f"Stale trajectory cache is missing {missing_keys}: {trajectory_cache}")
    print("loaded:", trajectory_cache)
else:
    torch.cuda.synchronize(DEVICE)
    started = time.perf_counter()
    raw_result = sim.simulate(
        B=B,
        burn_in=burn_in,
        n_steps=n_steps,
        save_interval=save_interval,
        show_progress=True,
        save_diagnostics=False,
        save_occupancy=True,
        save_exact_medium_ep=True,
        save_ep_maps=True,
    )
    torch.cuda.synchronize(DEVICE)
    elapsed = time.perf_counter() - started
    result = {key: raw_result[key] for key in tensor_keys}
    trajectory_was_generated = True
    print(
        f"simulated in {elapsed:.2f} s "
        f"({B * (burn_in + n_steps) / max(elapsed, 1e-12):.1f} ensemble sweeps/s)"
    )
    del raw_result
    torch.cuda.empty_cache()

# A valid thermodynamic trajectory must have finite exact EP everywhere.
if not bool(torch.isfinite(result["exact_medium_ep"]).all()):
    raise RuntimeError(
        "Non-finite exact EP indicates an invalid implementation or stale cache. "
        "Restart the kernel, then set REUSE_TRAJECTORY=False once to regenerate it."
    )
if not bool(torch.isfinite(result["exact_medium_ep_maps"]).all()):
    raise RuntimeError("Non-finite exact local EP map.")
assert int(result["occupancy"].min()) >= 0
assert int(result["occupancy"].max()) <= 1
assert bool((result["occupancy"].sum(dim=(-2, -1)) == params.N).all())
torch.testing.assert_close(
    result["exact_medium_ep_maps"].sum(dim=(-2, -1)).T,
    result["exact_medium_ep"],
    rtol=2.0e-5,
    atol=2.0e-4,
)
torch.testing.assert_close(
    result["exact_active_medium_ep"] + result["exact_wca_medium_ep"],
    result["exact_medium_ep"],
    rtol=2.0e-5,
    atol=2.0e-4,
)

if trajectory_was_generated:
    # A killed process can leave the temporary file, but never a partial file
    # at the canonical cache path.
    temporary_cache = trajectory_cache.with_suffix(trajectory_cache.suffix + ".tmp")
    torch.save(result, temporary_cache)
    os.replace(temporary_cache, trajectory_cache)
    print("saved:", trajectory_cache)

# Lightweight stationarity diagnostics across every replica.  These are not a
# substitute for longer seed-group/block analysis, but expose obvious drift.
exact_rate_matrix = result["exact_medium_ep_rate"].numpy()
half = max(1, exact_rate_matrix.shape[1] // 2)
early_ep_rate = exact_rate_matrix[:, :half].mean(axis=1)
late_ep_rate = exact_rate_matrix[:, half:].mean(axis=1)
overall_ep_rate = exact_rate_matrix.mean(axis=1)
relative_ep_drift = np.abs(late_ep_rate - early_ep_rate) / np.maximum(
    np.abs(overall_ep_rate), 1.0e-12
)
initial_mips = [
    sim.mips_summary(result["occupancy"][0, replica], include_coarse=False)
    for replica in range(B)
]
final_mips = [
    sim.mips_summary(result["occupancy"][-1, replica], include_coarse=False)
    for replica in range(B)
]
stationarity = {
    "early_exact_medium_ep_rate": early_ep_rate.tolist(),
    "late_exact_medium_ep_rate": late_ep_rate.tolist(),
    "relative_exact_ep_drift": relative_ep_drift.tolist(),
    "initial_largest_site_cluster_fraction": [
        item["largest_site_cluster_fraction"] for item in initial_mips
    ],
    "final_largest_site_cluster_fraction": [
        item["largest_site_cluster_fraction"] for item in final_mips
    ],
    "initial_low_k_ratio": [item["low_k_ratio"] for item in initial_mips],
    "final_low_k_ratio": [item["low_k_ratio"] for item in final_mips],
}
if float(relative_ep_drift.max()) > 0.25:
    warnings.warn(
        "At least one replica has >25% early/late mean EP-rate drift. "
        "Increase burn_in/production and use a new cache before interpreting NESS rates."
    )

print("sites:", tuple(result["sites"].shape))
print("occupancy:", tuple(result["occupancy"].shape))
print("exact total EP:", tuple(result["exact_medium_ep"].shape))
print("exact local EP:", tuple(result["exact_medium_ep_maps"].shape))
print("mean exact medium EP rate:", result["exact_medium_ep_rate"].mean(dim=1).numpy())
print("relative early/late EP-rate drift:", relative_ep_drift)
print("initial largest-cluster fractions:", stationarity["initial_largest_site_cluster_fraction"])
print("final largest-cluster fractions:", stationarity["final_largest_site_cluster_fraction"])
print("initial low-k ratios:", stationarity["initial_low_k_ratio"])
print("final low-k ratios:", stationarity["final_low_k_ratio"])

## 3. Observed fields and ensemble split

Arrays from the simulator are time-major.  KNEEP uses
`[ensemble, time, component, x, y]`.  The exact map is changed from
`[time_pair, ensemble, x, y]` to `[ensemble, time_pair, x, y]` with the same
pair index.

The split is by replica, not by neighboring frames: replicas 0--79 are used for
training, 80--99 for validation, and replica 100 is the single held-out test
trajectory.  Its exact EP never enters training or checkpoint selection. Fused
CUDA uses independent initial states and uniform draws across `B`, but shares
each sweep's particle permutation. Use a new simulator seed/cache for fully
independent repeat-level error bars.

In [ ]:
def encode_observed_fields(simulation_result, mode):
    occupancy = simulation_result["occupancy"].float()  # [T,B,G,G]
    if mode == "occupancy":
        return occupancy.permute(1, 0, 2, 3).unsqueeze(2).contiguous()
    if mode != "occupancy_polarization":
        raise ValueError(f"Unknown OBSERVATION_MODE={mode!r}")

    sites = simulation_result["sites"].long()            # [T,B,N,2]
    theta = simulation_result["theta"].float()           # [T,B,N]
    T, B_here, _, _ = sites.shape
    linear = sites[..., 0] * GRID_SIZE + sites[..., 1]
    pol_x = torch.zeros((T, B_here, GRID_SIZE * GRID_SIZE), dtype=torch.float32)
    pol_y = torch.zeros_like(pol_x)
    pol_x.scatter_add_(2, linear, torch.cos(theta))
    pol_y.scatter_add_(2, linear, torch.sin(theta))
    pol_x = pol_x.view(T, B_here, GRID_SIZE, GRID_SIZE)
    pol_y = pol_y.view(T, B_here, GRID_SIZE, GRID_SIZE)
    fields = torch.stack((occupancy, pol_x, pol_y), dim=2)
    return fields.permute(1, 0, 2, 3, 4).contiguous()


observed = encode_observed_fields(result, OBSERVATION_MODE)
# The cache already contains these raw tensors; the remaining notebook uses the
# encoded fields, so release roughly 0.8 GiB at B=101.
result.pop("sites")
result.pop("theta")
gt_local_all = result["exact_medium_ep_maps"].permute(1, 0, 2, 3).contiguous()
gt_total_all = result["exact_medium_ep"]

assert observed.shape[:2] == (B, n_steps // save_interval + 1)
assert observed.shape[2:] == (opt.n_components, GRID_SIZE, GRID_SIZE)
assert gt_local_all.shape[:2] == (B, observed.shape[1] - 1)
torch.testing.assert_close(
    gt_local_all.sum(dim=(-2, -1)), gt_total_all, rtol=2.0e-5, atol=2.0e-4
)

train_ensemble_ids = torch.arange(0, N_TRAIN_ENSEMBLES)
val_ensemble_ids = torch.arange(
    N_TRAIN_ENSEMBLES,
    N_TRAIN_ENSEMBLES + N_VALIDATION_ENSEMBLES,
)
test_ensemble_ids = torch.tensor([B - 1])
assert len(train_ensemble_ids) == 80
assert len(val_ensemble_ids) == 20
assert test_ensemble_ids.tolist() == [100]

# Contiguous slices stay as views and avoid duplicating the large field tensor.
train_video = observed[:N_TRAIN_ENSEMBLES]
val_video = observed[
    N_TRAIN_ENSEMBLES : N_TRAIN_ENSEMBLES + N_VALIDATION_ENSEMBLES
]
test_video = observed[-N_TEST_ENSEMBLES:]
gt_local_test = gt_local_all[-N_TEST_ENSEMBLES:]
gt_total_test = gt_total_all[-N_TEST_ENSEMBLES:]

channel_mean = train_video.mean(dim=(0, 1, 3, 4), keepdim=True)
channel_std = train_video.std(dim=(0, 1, 3, 4), keepdim=True).clamp_min(1.0e-6)


def normalize_pairs(x):
    return (x - channel_mean) / channel_std


print("observed:", tuple(observed.shape))
print("train/val/test replicas:", train_ensemble_ids.tolist(), val_ensemble_ids.tolist(), test_ensemble_ids.tolist())
print("channel mean:", channel_mean.flatten().numpy())
print("channel std:", channel_std.flatten().numpy())

## 4. KNEEP_2D model and unsupervised objective

In [ ]:
# Make model initialization and random pair batches independent of whether the
# trajectory was simulated or loaded from cache.
torch.manual_seed(opt.seed)
np.random.seed(opt.seed)
torch.cuda.manual_seed_all(opt.seed)
if opt.model_type == "MultiScaleK_2DF":
    model = MultiScaleK_2DF(opt).to(opt.device)
elif opt.model_type == "MultiScaleK_2D":
    model = MultiScaleK_2D(opt).to(opt.device)
else:
    raise ValueError(f"Unknown model type: {opt.model_type!r}")
optimizer = torch.optim.AdamW(model.parameters(), lr=opt.lr, weight_decay=opt.wd)
print(f"model parameters: {sum(p.numel() for p in model.parameters()):,}")


def make_pair_batch(video, ensemble_idx, time_idx):
    x0 = video[ensemble_idx, time_idx]
    x1 = video[ensemble_idx, time_idx + 1]
    x = torch.stack((x0, x1), dim=1)
    x = normalize_pairs(x)
    if opt.n_components == 1:
        x = x.squeeze(2)  # MultiScaleK_2DF expects [batch,time,x,y] for one component.
    return x.to(opt.device, non_blocking=True)


def random_pair_batch(video, batch_size):
    ensemble_idx = torch.randint(video.shape[0], (batch_size,))
    time_idx = torch.randint(video.shape[1] - 1, (batch_size,))
    return make_pair_batch(video, ensemble_idx, time_idx)


def ordered_pair_batches(video, batch_size):
    n_pairs = video.shape[1] - 1
    flat = torch.arange(video.shape[0] * n_pairs)
    for start in range(0, len(flat), batch_size):
        ids = flat[start : start + batch_size]
        yield ids // n_pairs, ids % n_pairs


def alpha_neep_loss(score, alpha):
    if alpha == 0:
        return (-score + torch.exp(-score) - 1.0).mean()
    return (
        -(torch.exp(alpha * score) - 1.0) / alpha
        + (torch.exp(-(1.0 + alpha) * score) - 1.0) / (1.0 + alpha)
    ).mean()


def spatial_mean_score(x):
    """Match the established no-angle KNEEP convention exactly."""
    return model(x).sum(dim=1)


@torch.no_grad()
def evaluate_objective(video):
    model.eval()
    total = 0.0
    count = 0
    for ensemble_idx, time_idx in ordered_pair_batches(video, opt.eval_batch_size):
        x = make_pair_batch(video, ensemble_idx, time_idx)
        score = spatial_mean_score(x)
        batch_loss = alpha_neep_loss(score, opt.alpha)
        total += float(batch_loss) * len(x)
        count += len(x)
    return total / max(count, 1)


# Model API and antisymmetry invariant check before training.
model.eval()
with torch.no_grad():
    x_check = random_pair_batch(train_video, min(2, opt.train_batch_size))
    j_check = model(x_check)
    maps_check = model(x_check, return_maps=True)
    assert j_check.shape == (len(x_check), opt.max_distance + 1)
    assert maps_check.shape == (len(x_check), opt.max_distance + 1, GRID_SIZE, GRID_SIZE)
    torch.testing.assert_close(j_check, maps_check.mean(dim=(-2, -1)), rtol=2e-5, atol=2e-6)
    torch.testing.assert_close(
        model(torch.flip(x_check, dims=[1]), return_maps=True),
        -maps_check,
        rtol=2e-5,
        atol=2e-6,
    )
print("KNEEP shape/antisymmetry check passed")

In [ ]:
best_val_loss = float("inf")
history_iter = []
history_train = []
history_val = []

for iteration in tqdm(range(1, opt.n_iter + 1), desc="KNEEP training"):
    model.train()
    x = random_pair_batch(train_video, opt.train_batch_size)
    score = spatial_mean_score(x)
    loss = alpha_neep_loss(score, opt.alpha)
    if not bool(torch.isfinite(loss)):
        raise FloatingPointError(f"non-finite training loss at iteration {iteration}")

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), opt.clip_norm)
    optimizer.step()

    if iteration == 1 or iteration % opt.record_freq == 0 or iteration == opt.n_iter:
        val_loss = evaluate_objective(val_video)
        if not math.isfinite(val_loss):
            raise FloatingPointError(f"non-finite validation loss at iteration {iteration}")
        history_iter.append(iteration)
        history_train.append(float(loss.detach().cpu()))
        history_val.append(val_loss)

        state = {
            "settings": vars(opt),
            "state_dict": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "iteration": iteration,
            "train_loss": history_train[-1],
            "val_loss": val_loss,
            "channel_mean": channel_mean,
            "channel_std": channel_std,
            "physics": physics,
            "trajectory_config": trajectory_config,
            "observation_mode": OBSERVATION_MODE,
        }
        torch.save(state, current_checkpoint_path)
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(state, best_checkpoint_path)

        print(
            f"iter={iteration:5d} train={history_train[-1]:+.6e} "
            f"val={val_loss:+.6e} best={best_val_loss:+.6e}"
        )

checkpoint = torch.load(best_checkpoint_path, map_location=opt.device, weights_only=False)
model.load_state_dict(checkpoint["state_dict"])
model.eval()
print(f"loaded best checkpoint from iteration {checkpoint['iteration']}")
if checkpoint["val_loss"] >= -1.0e-6:
    warnings.warn(
        "Best validation objective did not improve meaningfully over the zero-score baseline (0). "
        "Treat downstream EP plots as a failed-learning diagnostic."
    )

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(history_iter, history_train, "o-", label="train")
ax.plot(history_iter, history_val, "o-", label="validation")
ax.set_xlabel("iteration")
ax.set_ylabel("alpha-NEEP objective")
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()
fig.savefig(current_result_folder / "training_loss.png", dpi=160)
plt.show()

## 5. Held-out prediction and exact-map alignment

`MultiScaleK_2DF` returns the spatial **mean** of each branch map.  This notebook
uses the lattice-site convention from the existing lattice notebooks:

```text
predicted local increment at site = sum_k map_k
predicted total increment         = sum_sites predicted local increment
                                  = G^2 * sum_k J_k
```

The exact simulator map is already an increment per departure site, so no
$d\ell^2$ factor is inserted.  Dividing either map by `dt_saved` gives the
per-site rate.  If a physical-area density is desired, divide both maps by
$d\ell^2$ as a separate unit conversion.

To remain directly comparable with the established no-angle notebook, α-NEEP
is trained on the spatial mean of the learned local map and the raw learned
map is used in every local comparison.  The reported total is its site sum,
which is `G^2` times that spatial-mean score.  This is the repository's KNEEP
plotting convention, rather than a claim that the learned two-frame map is a
pointwise microscopic-EP label.

In [ ]:
@torch.no_grad()
def predict_local_maps(video):
    model.eval()
    chunks = []
    for ensemble_idx, time_idx in tqdm(
        ordered_pair_batches(video, opt.eval_batch_size),
        total=math.ceil(video.shape[0] * (video.shape[1] - 1) / opt.eval_batch_size),
        desc="held-out maps",
    ):
        x = make_pair_batch(video, ensemble_idx, time_idx)
        branch_maps = model(x, return_maps=True)
        chunks.append(branch_maps.detach().cpu())
    return torch.cat(chunks, dim=0)


pred_branch_maps = predict_local_maps(test_video)  # [Npair,K,G,G]
expected_test_pairs = test_video.shape[0] * (test_video.shape[1] - 1)
assert pred_branch_maps.shape == (
    expected_test_pairs,
    opt.max_distance + 1,
    GRID_SIZE,
    GRID_SIZE,
)
assert bool(torch.isfinite(pred_branch_maps).all())
pred_branch_inc_maps = pred_branch_maps
pred_local_inc = pred_branch_inc_maps.sum(dim=1)
pred_total_inc = pred_local_inc.sum(dim=(-2, -1))

gt_local_inc = gt_local_test.reshape(-1, GRID_SIZE, GRID_SIZE)
gt_total_inc = gt_total_test.reshape(-1)
torch.testing.assert_close(
    gt_local_inc.sum(dim=(-2, -1)), gt_total_inc, rtol=2.0e-5, atol=2.0e-4
)
torch.testing.assert_close(
    pred_branch_inc_maps.mean(dim=(-2, -1)).sum(dim=1) * GRID_SIZE**2,
    pred_total_inc,
    rtol=2.0e-5,
    atol=2.0e-4,
)
torch.testing.assert_close(
    spatial_mean_score(make_pair_batch(test_video, torch.arange(min(2, len(test_video))), torch.zeros(min(2, len(test_video)), dtype=torch.long))).detach().cpu() * GRID_SIZE**2,
    pred_total_inc[: min(2, len(test_video))],
    rtol=2.0e-5,
    atol=2.0e-4,
)

pred_total_rate = pred_total_inc.numpy() / dt_saved
gt_total_rate = gt_total_inc.numpy() / dt_saved
pred_local_rate = pred_local_inc.numpy() / dt_saved
gt_local_rate = gt_local_inc.numpy() / dt_saved
assert np.isfinite(pred_total_rate).all()
assert np.isfinite(gt_total_rate).all()
assert np.isfinite(pred_local_rate).all()
assert np.isfinite(gt_local_rate).all()
if float(pred_total_rate.mean()) <= 0.0:
    warnings.warn("Held-out mean KNEEP score is non-positive; training did not resolve positive irreversibility.")

# Unit audit.  The simulator's map is microscopic hop-wise medium EP, whereas
# KNEEP estimates an observed two-frame score.  The latter is not a pointwise
# label.  The identities below follow the same mean-map/site-sum convention as
# the no-angle notebook and catch stale/manual plotting cells.
raw_pred_total_inc = pred_branch_maps.sum(dim=(1, 2, 3))
score_from_model = torch.cat(
    [
        spatial_mean_score(
            make_pair_batch(
                test_video,
                ensemble_idx,
                time_idx,
            )
        ).detach().cpu()
        for ensemble_idx, time_idx in ordered_pair_batches(test_video, opt.eval_batch_size)
    ]
)
torch.testing.assert_close(raw_pred_total_inc, pred_total_inc, rtol=2.0e-5, atol=2.0e-4)
torch.testing.assert_close(
    score_from_model * GRID_SIZE**2, pred_total_inc, rtol=2.0e-5, atol=2.0e-4
)
np.testing.assert_allclose(
    gt_local_rate.sum(axis=(-2, -1)), gt_total_rate, rtol=2.0e-5, atol=2.0e-4
)
scale_audit = {
    "dt_saved": float(dt_saved),
    "grid_sites": int(GRID_SIZE**2),
    "model_score_to_total_factor": int(GRID_SIZE**2),
    "mean_true_microscopic_total_increment": float(gt_total_inc.mean()),
    "mean_true_microscopic_total_rate": float(gt_total_rate.mean()),
    "mean_true_microscopic_site_rate": float(gt_local_rate.mean()),
    "mean_predicted_observed_total_increment": float(pred_total_inc.mean()),
    "mean_predicted_observed_total_rate": float(pred_total_rate.mean()),
    "mean_predicted_observed_site_rate": float(pred_local_rate.mean()),
    "observed_to_microscopic_total_rate_ratio": float(
        pred_total_rate.mean() / gt_total_rate.mean()
    ),
    "true_local_rate_min": float(gt_local_rate.min()),
    "true_local_rate_max": float(gt_local_rate.max()),
    "predicted_local_rate_min": float(pred_local_rate.min()),
    "predicted_local_rate_max": float(pred_local_rate.max()),
}
print("scale audit:")
print(json.dumps(scale_audit, indent=2))


def comparison_metrics(target, prediction):
    target = np.asarray(target, dtype=np.float64)
    prediction = np.asarray(prediction, dtype=np.float64)
    if target.shape != prediction.shape:
        raise ValueError(f"metric shape mismatch: {target.shape} != {prediction.shape}")
    if not (np.isfinite(target).all() and np.isfinite(prediction).all()):
        raise ValueError("comparison_metrics received non-finite values")
    residual = prediction - target
    denom = np.sum((target - target.mean()) ** 2)
    correlation = (
        float(np.corrcoef(target, prediction)[0, 1])
        if len(target) > 1 and target.std() > 0 and prediction.std() > 0
        else float("nan")
    )
    return {
        "count": int(len(target)),
        "target_mean": float(target.mean()),
        "prediction_mean": float(prediction.mean()),
        "mean_ratio": float(prediction.mean() / target.mean()) if target.mean() != 0 else float("nan"),
        "mae": float(np.mean(np.abs(residual))),
        "rmse": float(np.sqrt(np.mean(residual**2))),
        "pearson_r": correlation,
        "raw_r2": float(1.0 - np.sum(residual**2) / denom) if denom > 0 else float("nan"),
    }


total_metrics = comparison_metrics(gt_total_rate, pred_total_rate)
test_objective = evaluate_objective(test_video)
print(json.dumps(total_metrics, indent=2))
print(f"held-out alpha-NEEP objective: {test_objective:+.6e} (zero-score baseline: 0)")

## 6. Total medium EP: held-out time series, cumulative value, and scatter

In [ ]:
def moving_average(values, window):
    values = np.asarray(values)
    if len(values) == 0:
        return values
    window = max(1, min(int(window), len(values)))
    return np.convolve(values, np.ones(window) / window, mode="same")


n_pairs_per_test_replica = test_video.shape[1] - 1
time_mid = result["times"][:-1].numpy() + 0.5 * dt_saved
time_end = result["times"][1:].numpy()
plot_slice = slice(0, n_pairs_per_test_replica)  # first held-out replica
smooth_window = min(100, max(1, n_pairs_per_test_replica // 10))

fig, axes = plt.subplots(3, 1, figsize=(12, 10), constrained_layout=True)
axes[0].plot(
    time_mid,
    moving_average(gt_total_rate[plot_slice], smooth_window),
    label=f"true microscopic (MA {smooth_window})",
)
axes[0].plot(
    time_mid,
    moving_average(pred_total_rate[plot_slice], smooth_window),
    label=f"KNEEP visible (MA {smooth_window})",
)
axes[0].set_ylabel("medium EP rate")
axes[0].set_title("Held-out total medium EP rate")
axes[0].legend()
axes[0].grid(alpha=0.25)

axes[1].plot(time_end, np.cumsum(gt_total_inc.numpy()[plot_slice]), label="true microscopic")
axes[1].plot(time_end, np.cumsum(pred_total_inc.numpy()[plot_slice]), label="KNEEP visible")
axes[1].set_ylabel("cumulative EP")
axes[1].legend()
axes[1].grid(alpha=0.25)

axes[2].scatter(gt_total_rate, pred_total_rate, s=9, alpha=0.35)
lo = float(min(gt_total_rate.min(), pred_total_rate.min()))
hi = float(max(gt_total_rate.max(), pred_total_rate.max()))
axes[2].plot([lo, hi], [lo, hi], "k--", lw=1, label="identity")
axes[2].set_xlabel("true microscopic medium EP rate")
axes[2].set_ylabel("KNEEP visible EP rate")
axes[2].set_title(
    f"Held-out pairs: r={total_metrics['pearson_r']:.3f}, "
    f"raw R2={total_metrics['raw_r2']:.3f}"
)
axes[2].legend()
axes[2].grid(alpha=0.25)

fig.savefig(current_result_folder / "total_medium_ep_comparison.png", dpi=160)
plt.show()

## 7. True versus learned local medium EP

The true map uses the simulator's **departure-site gauge**.  The learned map is
localized by the convolution center and may choose a different gauge while
preserving a similar total.  Therefore the notebook shows a single interval,
the held-out time average, residuals, and pixelwise metrics; the total and
cumulative comparisons above remain the gauge-robust diagnostics.  Pixel R2 is
only a gauge-dependent localization diagnostic, and the absolute time-mean map
can also reflect pinning or drift of the periodic MIPS cluster.

In [ ]:
sample_index = n_pairs_per_test_replica // 2
test_replica_original = int(test_ensemble_ids[0])
occupancy_sample = result["occupancy"][sample_index, test_replica_original].numpy()
true_sample = gt_local_rate[sample_index]
pred_sample = pred_local_rate[sample_index]
residual_sample = pred_sample - true_sample

vmax = max(float(np.abs(true_sample).max()), float(np.abs(pred_sample).max()), 1.0e-12)
residual_vmax = max(float(np.abs(residual_sample).max()), 1.0e-12)

fig, axes = plt.subplots(1, 4, figsize=(18, 4), constrained_layout=True)
im0 = axes[0].imshow(occupancy_sample.T, origin="lower", cmap="Greys", vmin=0, vmax=1)
axes[0].set_title(f"occupancy, pair {sample_index}")
fig.colorbar(im0, ax=axes[0], shrink=0.8)

im1 = axes[1].imshow(true_sample.T, origin="lower", cmap="RdBu_r", vmin=-vmax, vmax=vmax)
axes[1].set_title("true local medium EP rate")
fig.colorbar(im1, ax=axes[1], shrink=0.8)

im2 = axes[2].imshow(pred_sample.T, origin="lower", cmap="RdBu_r", vmin=-vmax, vmax=vmax)
axes[2].set_title("KNEEP local visible EP rate")
fig.colorbar(im2, ax=axes[2], shrink=0.8)

im3 = axes[3].imshow(
    residual_sample.T,
    origin="lower",
    cmap="RdBu_r",
    vmin=-residual_vmax,
    vmax=residual_vmax,
)
axes[3].set_title("KNEEP - true")
fig.colorbar(im3, ax=axes[3], shrink=0.8)
for ax in axes:
    ax.set_xticks([])
    ax.set_yticks([])

fig.savefig(current_result_folder / "local_medium_ep_single_pair.png", dpi=170)
plt.show()

single_local_metrics = comparison_metrics(true_sample.ravel(), pred_sample.ravel())
print("single-pair pixel metrics:")
print(json.dumps(single_local_metrics, indent=2))

In [ ]:
true_mean_map = gt_local_rate[:n_pairs_per_test_replica].mean(axis=0)
pred_mean_map = pred_local_rate[:n_pairs_per_test_replica].mean(axis=0)
mean_residual = pred_mean_map - true_mean_map
mean_local_metrics = comparison_metrics(true_mean_map.ravel(), pred_mean_map.ravel())

# The printed pixel means and plotted fields must refer to exactly the same
# arrays.  In particular, a true-map mean above a plot's colour limit signals
# clipping (or that an old notebook cell was run), not a simulator scale error.
np.testing.assert_allclose(true_mean_map.mean(), gt_local_rate.mean(), rtol=2.0e-6, atol=2.0e-6)
np.testing.assert_allclose(pred_mean_map.mean(), pred_local_rate.mean(), rtol=2.0e-6, atol=2.0e-6)

vmax = max(float(np.abs(true_mean_map).max()), float(np.abs(pred_mean_map).max()), 1.0e-12)
residual_vmax = max(float(np.abs(mean_residual).max()), 1.0e-12)
fig, axes = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)
im0 = axes[0].imshow(true_mean_map.T, origin="lower", cmap="RdBu_r", vmin=-vmax, vmax=vmax)
axes[0].set_title("time-mean true local rate")
fig.colorbar(im0, ax=axes[0], shrink=0.8)
im1 = axes[1].imshow(pred_mean_map.T, origin="lower", cmap="RdBu_r", vmin=-vmax, vmax=vmax)
axes[1].set_title("time-mean KNEEP local rate")
fig.colorbar(im1, ax=axes[1], shrink=0.8)
im2 = axes[2].imshow(
    mean_residual.T,
    origin="lower",
    cmap="RdBu_r",
    vmin=-residual_vmax,
    vmax=residual_vmax,
)
axes[2].set_title(
    f"mean residual (r={mean_local_metrics['pearson_r']:.3f})"
)
fig.colorbar(im2, ax=axes[2], shrink=0.8)
for ax in axes:
    ax.set_xticks([])
    ax.set_yticks([])

fig.savefig(current_result_folder / "local_medium_ep_time_mean.png", dpi=170)
plt.show()

print("time-mean pixel metrics:")
print(json.dumps(mean_local_metrics, indent=2))

## 8. K-shell contribution spectrum and saved diagnostics

In [ ]:
# Branch maps are converted to physical increments before summing/dividing by time.
branch_total_rates = pred_branch_inc_maps.sum(dim=(-2, -1)).numpy() / dt_saved
mean_branch_rate = branch_total_rates.mean(axis=0)
distances = np.arange(0 if opt.include_k0 else 1, opt.max_distance + 1)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), constrained_layout=True)
axes[0].bar(distances, mean_branch_rate, color="steelblue")
axes[0].set_xlabel("exclusive Euclidean shell k")
axes[0].set_ylabel("mean predicted EP rate")
axes[0].set_title("KNEEP shell spectrum (descriptive pair mean)")
axes[0].set_xticks(distances)
axes[0].grid(axis="y", alpha=0.25)

axes[1].plot(distances, np.cumsum(mean_branch_rate), "o-", color="darkorange")
axes[1].axhline(gt_total_rate.mean(), color="black", ls="--", label="true microscopic total")
axes[1].set_xlabel("maximum included Euclidean shell k")
axes[1].set_ylabel("cumulative predicted EP rate")
axes[1].set_title("Cumulative visible EP by range")
axes[1].set_xticks(distances)
axes[1].legend()
axes[1].grid(alpha=0.25)

fig.savefig(current_result_folder / "kneep_shell_spectrum.png", dpi=160)
plt.show()

summary = {
    "model": opt.model_type,
    "observation_mode": OBSERVATION_MODE,
    "physics": {**physics, "actual_phi": params.phi, "dl": params.dl, "Pe": params.Pe},
    "trajectory_config": trajectory_config,
    "trajectory_cache_id": cache_id,
    "sampler_source_sha256": sampler_source_sha256,
    "split": {
        "train_ensemble_ids": train_ensemble_ids.tolist(),
        "validation_ensemble_ids": val_ensemble_ids.tolist(),
        "test_ensemble_ids": test_ensemble_ids.tolist(),
        "pairs_per_ensemble": int(n_pairs_per_test_replica),
    },
    "best_iteration": int(checkpoint["iteration"]),
    "best_validation_loss": float(checkpoint["val_loss"]),
    "held_out_objective": float(test_objective),
    "dt_saved": float(dt_saved),
    "scale_audit": scale_audit,
    "stationarity_diagnostics": stationarity,
    "total_rate_metrics": total_metrics,
    "single_pair_local_metrics": single_local_metrics,
    "time_mean_local_metrics": mean_local_metrics,
    "local_metric_note": "Gauge-dependent diagnostic; not a pairwise thermodynamic equality or bound.",
    "mean_branch_rate": mean_branch_rate.tolist(),
    "mean_true_total_rate": float(gt_total_rate.mean()),
    "mean_predicted_total_rate": float(pred_total_rate.mean()),
    "mean_true_active_medium_ep_rate": float(
        result["exact_active_medium_ep"][test_ensemble_ids].mean() / dt_saved
    ),
    "mean_true_wca_medium_ep_rate": float(
        result["exact_wca_medium_ep"][test_ensemble_ids].mean() / dt_saved
    ),
    "mean_accepted_hops_per_saved_interval": float(
        result["accepted_hops"][test_ensemble_ids].float().mean()
    ),
}


def json_safe(value):
    if isinstance(value, dict):
        return {str(key): json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(item) for item in value]
    if isinstance(value, np.generic):
        value = value.item()
    if isinstance(value, float) and not math.isfinite(value):
        return None
    return value


summary_for_json = json_safe(summary)
(current_result_folder / "summary.json").write_text(
    json.dumps(summary_for_json, indent=2, allow_nan=False), encoding="utf-8"
)
np.savez_compressed(
    current_result_folder / "held_out_ep_comparison.npz",
    test_ensemble_ids=test_ensemble_ids.numpy(),
    times_mid=time_mid,
    times_end=time_end,
    gt_total_increment=gt_total_inc.numpy(),
    pred_total_increment=pred_total_inc.numpy(),
    gt_total_rate=gt_total_rate,
    pred_total_rate=pred_total_rate,
    gt_local_rate=gt_local_rate,
    pred_local_rate=pred_local_rate,
    gt_time_mean_local_rate=true_mean_map,
    pred_time_mean_local_rate=pred_mean_map,
    mean_branch_rate=mean_branch_rate,
    history_iteration=np.asarray(history_iter),
    history_train_loss=np.asarray(history_train),
    history_validation_loss=np.asarray(history_val),
)

print(json.dumps(summary_for_json, indent=2, allow_nan=False))
print("saved diagnostics under:", current_result_folder)